In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers pydantic accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 144.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.9 MB/s eta 0:00:00


In [ ]:
import os
# Use Colab Secrets (left sidebar 🔑) instead of hardcoding
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [ ]:
!nvidia-smi
# Must show T4 with ~12GB VRAM


Thu Apr 23 10:34:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch, gc

try:
    model
    print("✅ Model already loaded — skipping.")
except NameError:
    from unsloth import FastLanguageModel

    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    print(f"Loading {MODEL_NAME}...")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        load_in_4bit=True,
        dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=4,
        lora_alpha=8,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    print("✅ Model ready.")
import torch, gc

try:
    model
    print("✅ Model already loaded — skipping.")
except NameError:
    from unsloth import FastLanguageModel

    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    print(f"Loading {MODEL_NAME}...")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        load_in_4bit=True,
        dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=4,
        lora_alpha=8,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    print("✅ Model ready.")


✅ Model already loaded — skipping.
✅ Model already loaded — skipping.


In [ ]:
import random
from collections import defaultdict

SENSOR_TYPES    = ["satellite", "drone", "radar"]
TARGET_TYPES    = ["strategic", "kinetic", "airspace"]
PRIORITY_REWARD = {3: 2.0, 2: 1.0, 1: 0.5}
MISSED_PENALTY  = -2.0
IDLE_PENALTY    = -2.0

CAPABILITY_MATRIX = {
    ("satellite", "strategic"): 0.95, ("satellite", "kinetic"): 0.40,
    ("satellite", "airspace"):  0.60, ("drone",     "kinetic"): 0.95,
    ("drone",     "strategic"): 0.30, ("drone",     "airspace"): 0.50,
    ("radar",     "airspace"):  0.95, ("radar",     "kinetic"):  0.65,
    ("radar",     "strategic"): 0.45,
}

def make_env(seed=0, max_steps=20):
    rng = random.Random(seed)
    sensors = [
        {"id": f"S{i+1}", "type": rng.choice(SENSOR_TYPES),
         "available": True, "range": rng.randint(100, 500)}
        for i in range(rng.randint(3, 5))
    ]
    return {"sensors": sensors, "max_steps": max_steps, "seed": seed, "_rng": rng}

def env_reset(env_state):
    rng = env_state["_rng"]
    targets = [
        {"id": f"T0_{i+1}", "priority": rng.randint(1, 3),
         "type": rng.choice(TARGET_TYPES), "active": True}
        for i in range(rng.randint(2, 4))
    ]
    for s in env_state["sensors"]:
        s["available"] = True
    env_state.update({"targets": targets, "current_step": 0, "total_reward": 0.0})
    return env_state

def env_step(env_state, sensor_id, target_id):
    targets = env_state["targets"]
    sensors = env_state["sensors"]
    step_reward = 0.0
    handled = False
    for s in sensors:
        if s["id"] == sensor_id and s["available"]:
            for t in targets:
                if t["id"] == target_id and t["active"]:
                    cap = CAPABILITY_MATRIX.get((s["type"], t.get("type", "strategic")), 0.0)
                    p   = t["priority"]
                    step_reward += (3.0 if cap >= 0.85 else 2.0) if p == 3 else PRIORITY_REWARD.get(p, 0.0)
                    t["active"] = False
                    s["available"] = False
                    handled = True
                    break
            break
    if not handled:
        step_reward += IDLE_PENALTY
    for t in [t for t in targets if t["active"] and t["priority"] == 3]:
        for s in [s for s in sensors if s["available"]]:
            if CAPABILITY_MATRIX.get((s["type"], t.get("type", "strategic")), 0.0) > 0.5:
                step_reward += MISSED_PENALTY
                break
    env_state["current_step"] += 1
    s_rng = random.Random(env_state["seed"] * 1234 + env_state["current_step"])
    env_state["targets"] = [
        {"id": f"T{env_state['current_step']}_{i+1}",
         "priority": s_rng.randint(1, 3),
         "type": s_rng.choice(TARGET_TYPES), "active": True}
        for i in range(s_rng.randint(2, 4))
    ]
    for s in env_state["sensors"]:
        s["available"] = True
    env_state["total_reward"] += step_reward
    done = env_state["current_step"] >= env_state["max_steps"]
    return env_state, step_reward, done

def greedy_select(env_state):
    available = [s for s in env_state["sensors"] if s["available"]]
    active    = sorted([t for t in env_state["targets"] if t["active"]],
                       key=lambda t: -t["priority"])
    if not available or not active:
        return None, None
    return available[0]["id"], active[0]["id"]

def compute_conflict_penalty(proposals):
    """Penalise only when 2+ DIFFERENT sensors assigned to SAME target."""
    by_target = defaultdict(set)
    for p in proposals:
        by_target[p["target_id"]].add(p["sensor_id"])
    return sum(-1.0 for sensors in by_target.values() if len(sensors) > 1)

def get_noise_prob(episode, total=50):
    """Decay conflict noise from 0.45 → 0.05 over training."""
    return 0.45 - (0.40 * (episode - 1) / max(total - 1, 1))

print("✅ Environment ready.")


✅ Environment ready.


In [ ]:
import json, random, datasets
from trl import GRPOTrainer, GRPOConfig

# ── Build prompt dataset (200 unique env states as prompts) ──────────────────
def make_prompt(seed):
    rng = random.Random(seed)
    sensors = [
        {"id": f"S{i+1}", "type": rng.choice(SENSOR_TYPES), "range": rng.randint(100,500)}
        for i in range(rng.randint(3, 5))
    ]
    targets = [
        {"id": f"T0_{i+1}", "priority": rng.randint(1,3), "type": rng.choice(TARGET_TYPES)}
        for i in range(rng.randint(2, 4))
    ]
    sensor_lines = "\n".join(f"  {s['id']} type={s['type']} range={s['range']}km" for s in sensors)
    target_lines = "\n".join(f"  {t['id']} priority={t['priority']} type={t['type']}" for t in targets)
    sensor_ids   = [s["id"] for s in sensors]
    target_ids   = [t["id"] for t in targets]
    return (
        f"You are an ISR sensor allocation agent.\n"
        f"Assign ONE sensor to ONE target. Priority 3=HIGH, 2=MED, 1=LOW.\n"
        f"Cover the highest priority target first.\n\n"
        f"Available sensors:\n{sensor_lines}\n\n"
        f"Active targets:\n{target_lines}\n\n"
        f"Respond ONLY with valid JSON using these exact IDs:\n"
        f"sensors={sensor_ids}  targets={target_ids}\n\n"
        f"{{\"sensor_id\": \"S1\", \"target_id\": \"T0_1\"}}"
    ), sensor_ids, target_ids, targets

# Store metadata alongside prompts for reward function
prompt_list   = []
metadata_list = []

for seed in range(200):
    prompt, sensor_ids, target_ids, targets = make_prompt(seed)
    prompt_list.append(prompt)
    metadata_list.append({
        "sensor_ids": sensor_ids,
        "target_ids": target_ids,
        "targets": targets,
        "seed": seed,
    })

train_dataset = datasets.Dataset.from_dict({"prompt": prompt_list})
print(f"✅ Dataset ready: {len(train_dataset)} prompts")

# ── Real reward function ──────────────────────────────────────────────────────
def arya_reward_func(completions, prompts, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        try:
            # Find JSON in completion
            text  = completion if isinstance(completion, str) else completion[0]
            start = text.find("{")
            end   = text.rfind("}") + 1
            if start == -1 or end == 0:
                rewards.append(-2.0)
                continue

            data      = json.loads(text[start:end])
            sensor_id = data.get("sensor_id", "")
            target_id = data.get("target_id", "")

            # Get metadata for this prompt
            idx  = i % len(metadata_list)
            meta = metadata_list[idx]

            # Validate IDs
            if sensor_id not in meta["sensor_ids"] or target_id not in meta["target_ids"]:
                rewards.append(-1.0)  # invalid IDs
                continue

            # Compute real env reward
            target = next((t for t in meta["targets"] if t["id"] == target_id), None)
            if target is None:
                rewards.append(-1.0)
                continue

            priority = target["priority"]
            # Find sensor type from prompt metadata
            env_s = env_reset(make_env(seed=meta["seed"]))
            sensor = next((s for s in env_s["sensors"] if s["id"] == sensor_id), None)
            if sensor is None:
                rewards.append(-1.0)
                continue

            cap = CAPABILITY_MATRIX.get((sensor["type"], target.get("type","strategic")), 0.0)

            # Real reward signal matching ARYA-X reward engine
            if priority == 3:
                reward = 3.0 if cap >= 0.85 else 2.0
            elif priority == 2:
                reward = 1.0
            else:
                reward = 0.5

            # Bonus for optimal capability match
            if cap >= 0.85:
                reward += 0.5

            rewards.append(float(reward))

        except Exception:
            rewards.append(-2.0)

    return rewards

print("✅ Reward function ready.")
print(f"   Reward range: -2.0 (invalid) → +3.5 (P3 optimal match)")


✅ Dataset ready: 200 prompts
✅ Reward function ready.
   Reward range: -2.0 (invalid) → +3.5 (P3 optimal match)


In [ ]:
# ── GRPO Config ───────────────────────────────────────────────────────────────
grpo_args = GRPOConfig(
    output_dir="./grpo_out",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch = 16
    max_steps=200,                    # 200 real gradient updates
    learning_rate=2e-5,
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_args,
    tokenizer=tokenizer,
    reward_funcs=arya_reward_func,
    train_dataset=train_dataset,
)

print("✅ GRPOTrainer ready.")
print(f"   Steps     : {grpo_args.max_steps}")
print(f"   Batch     : {grpo_args.per_device_train_batch_size} x {grpo_args.gradient_accumulation_steps} = {grpo_args.per_device_train_batch_size * grpo_args.gradient_accumulation_steps}")
print(f"   LR        : {grpo_args.learning_rate}")
print("\nStarting training — this will take ~30-60 min on T4...\n")

# ── Train ─────────────────────────────────────────────────────────────────────
train_result = grpo_trainer.train()

print("\n✅ Real GRPO training complete.")
print(f"   Loss : {train_result.training_loss:.4f}")


✅ GRPOTrainer ready.
   Steps     : 200
   Batch     : 8 x 2 = 16
   LR        : 2e-05

Starting training — this will take ~30-60 min on T4...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 2 | Total steps = 200
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 1,146,880 of 3,213,896,704 (0.04% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: Futur

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / arya_reward_func / mean,rewards / arya_reward_func / std
10,0.018564,-1.806250,0.479714,245.731250,148.700000,256.000000,0.912500,97.075000,71.900000,116.900000,0.000006,-1.806250,0.638322
20,0.020714,-1.684375,0.641958,245.462500,116.100000,256.000000,0.918750,83.716667,64.900000,103.600000,0.000010,-1.684375,0.814143
30,0.005512,-1.718750,0.638071,244.700000,112.900000,256.000000,0.925000,76.100000,61.700000,90.500000,0.000016,-1.718750,0.738983
40,0.010131,-1.659375,0.752348,245.568750,142.800000,256.000000,0.900000,130.866669,91.600000,161.500000,0.000018,-1.659375,0.866026
50,0.003278,-1.728125,0.672936,250.556250,191.100000,256.000000,0.931250,154.366667,139.900000,169.200000,0.000023,-1.728125,0.829480
60,0.011792,-1.600000,0.933948,251.781250,203.900000,256.000000,0.968750,33.366667,24.700000,41.900000,0.000029,-1.600000,1.033027
70,0.014591,-1.765625,0.506901,246.087500,138.600000,256.000000,0.912500,103.983334,61.800000,143.900000,0.000061,-1.765625,0.594428
80,0.005331,-1.593750,0.836083,241.306250,112.200000,256.000000,0.906250,102.475000,86.600000,118.600000,0.000115,-1.593750,0.929971
90,0.022303,-1.568750,0.748286,246.593750,139.900000,256.000000,0.925000,124.200000,114.300000,134.100000,0.000105,-1.568750,0.874358
100,-0.001010,-1.675000,0.687488,241.756250,103.900000,256.000000,0.881250,140.233334,103.900000,180.300000,0.000130,-1.675000,0.809303


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1


✅ Real GRPO training complete.
   Loss : 0.0060


In [ ]:
# ── Evaluate trained model ────────────────────────────────────────────────────
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def evaluate_trained_model(n_samples=50):
    """Run trained model on env prompts, measure reward and conflict rate."""
    total_reward  = 0.0
    conflicts     = 0
    valid_outputs = 0

    for seed in range(n_samples):
        prompt, sensor_ids, target_ids, targets = make_prompt(seed)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,       # greedy decode for eval
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        text  = tokenizer.decode(outputs[0], skip_special_tokens=True)
        start = text.rfind("{")
        end   = text.rfind("}") + 1

        try:
            data      = json.loads(text[start:end])
            sensor_id = data.get("sensor_id","")
            target_id = data.get("target_id","")

            if sensor_id not in sensor_ids or target_id not in target_ids:
                conflicts += 1
                total_reward -= 1.0
                continue

            target   = next((t for t in targets if t["id"] == target_id), None)
            env_s    = env_reset(make_env(seed=seed))
            sensor   = next((s for s in env_s["sensors"] if s["id"] == sensor_id), None)

            if target and sensor:
                cap      = CAPABILITY_MATRIX.get((sensor["type"], target.get("type","strategic")), 0.0)
                priority = target["priority"]
                reward   = (3.0 if cap >= 0.85 else 2.0) if priority == 3 else {2:1.0,1:0.5}.get(priority,0.0)
                total_reward  += reward
                valid_outputs += 1
            else:
                conflicts += 1

        except Exception:
            conflicts += 1
            total_reward -= 2.0

    conflict_rate      = conflicts / n_samples
    coordination_score = 1.0 - conflict_rate
    avg_reward         = total_reward / n_samples

    return {
        "avg_reward":         round(avg_reward, 3),
        "conflict_rate":      round(conflict_rate, 3),
        "coordination_score": round(coordination_score, 3),
        "valid_outputs":      valid_outputs,
        "n_samples":          n_samples,
    }

print("Evaluating trained model on 50 samples...")
results = evaluate_trained_model(n_samples=50)

print(f"\n{'='*45}")
print(f"  TRAINED MODEL RESULTS")
print(f"{'='*45}")
print(f"  avg_reward         : {results['avg_reward']}")
print(f"  conflict_rate      : {results['conflict_rate']}")
print(f"  coordination_score : {results['coordination_score']}")
print(f"  valid_outputs      : {results['valid_outputs']}/{results['n_samples']}")
print(f"{'='*45}")


In [ ]:
import os, shutil

# Save metrics
os.makedirs("logs", exist_ok=True)
metrics = {
    "training_loss":      train_result.training_loss,
    "steps":              grpo_args.max_steps,
    "conflict_rate":      results["conflict_rate"],
    "coordination_score": results["coordination_score"],
    "avg_reward":         results["avg_reward"],
    "model":              "Llama-3.2-3B-Instruct",
    "method":             "GRPO + LoRA r=4",
}
with open("logs/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("✅ Saved: logs/training_metrics.json")

# Save checkpoint
os.makedirs("checkpoints/arya_x_lora", exist_ok=True)
model.save_pretrained("checkpoints/arya_x_lora")
tokenizer.save_pretrained("checkpoints/arya_x_lora")
print("✅ Saved: checkpoints/arya_x_lora/")